In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import py3Dmol
import os
from rdkit import Chem
from rdkit.Chem import AllChem
import re
from matplotlib.widgets import Slider
import matplotlib.patches as patches

class InsulinDenaturationSimulator:
    def __init__(self):
        """
        Simulador de desnaturalización de insulina
        """
        # Secuencia simplificada de insulina humana (cadena A y B)
        self.insulin_sequence_A = "GIVEQCCTSICSLYQLENYCN"  # Cadena A
        self.insulin_sequence_B = "FVNQHLCGSHLVEALYLVCGERGFFYTPKT"  # Cadena B
        
        # Parámetros de simulación
        self.temperature = 37.0  # Temperatura inicial en °C
        self.denaturation_threshold = 60.0  # Temperatura de desnaturalización
        self.native_coords = None
        self.denatured_coords = None
        self.current_coords = None
        
        # Generar coordenadas 3D iniciales
        self.generate_initial_structure()
        
    def generate_initial_structure(self):
        """
        Genera estructura 3D inicial de la insulina (estado nativo)
        """
        # Número total de residuos
        total_residues = len(self.insulin_sequence_A) + len(self.insulin_sequence_B)
        
        # Generar estructura alfa-hélice para el estado nativo
        self.native_coords = self.generate_alpha_helix(total_residues)
        
        # Generar estructura desplegada para el estado desnaturalizado
        self.denatured_coords = self.generate_random_coil(total_residues)
        
        # Coordenadas actuales (inicialmente nativas)
        self.current_coords = self.native_coords.copy()
        
    def generate_alpha_helix(self, n_residues, radius=3.0, pitch=1.5):
        """
        Genera coordenadas para una estructura alfa-hélice
        """
        coords = np.zeros((n_residues, 3))
        
        for i in range(n_residues):
            angle = i * 2 * np.pi / 3.6  # 3.6 residuos por vuelta
            coords[i, 0] = radius * np.cos(angle)
            coords[i, 1] = radius * np.sin(angle)
            coords[i, 2] = i * pitch
            
        return coords
    
    def generate_random_coil(self, n_residues, spread=15.0):
        """
        Genera coordenadas para una estructura desplegada (random coil)
        """
        coords = np.zeros((n_residues, 3))
        
        # Camino aleatorio con restricciones de distancia
        coords[0] = [0, 0, 0]
        
        for i in range(1, n_residues):
            # Dirección aleatoria
            direction = np.random.randn(3)
            direction = direction / np.linalg.norm(direction)
            
            # Distancia entre residuos consecutivos
            bond_length = 3.8 + np.random.normal(0, 0.5)
            coords[i] = coords[i-1] + direction * bond_length
            
        return coords
    
    def calculate_denaturation_fraction(self, temperature):
        """
        Calcula la fracción de desnaturalización basada en la temperatura
        Usando una función sigmoidal para transición cooperativa
        """
        # Parámetros de la transición
        Tm = self.denaturation_threshold  # Temperatura de fusión
        steepness = 0.2  # Factor de cooperatividad
        
        fraction = 1 / (1 + np.exp(-(temperature - Tm) * steepness))
        return fraction
    
    def update_structure(self, temperature):
        """
        Actualiza la estructura basada en la temperatura
        """
        self.temperature = temperature
        denat_fraction = self.calculate_denaturation_fraction(temperature)
        
        # Interpolación lineal entre estructura nativa y desnaturalizada
        self.current_coords = ((1 - denat_fraction) * self.native_coords + 
                              denat_fraction * self.denatured_coords)
        
        return denat_fraction
    
    def calculate_rmsd(self):
        """
        Calcula el RMSD con respecto a la estructura nativa
        """
        diff = self.current_coords - self.native_coords
        rmsd = np.sqrt(np.mean(np.sum(diff**2, axis=1)))
        return rmsd
    
    def plot_3d_structure(self, figsize=(12, 10)):
        """
        Visualiza la estructura 3D actual
        """
        fig = plt.figure(figsize=figsize)
        ax = fig.add_subplot(111, projection='3d')
        
        # Colores según la secuencia (cadena A vs cadena B)
        n_chain_A = len(self.insulin_sequence_A)
        colors = ['blue'] * n_chain_A + ['red'] * len(self.insulin_sequence_B)
        
        # Plot de los puntos
        for i, (coord, color) in enumerate(zip(self.current_coords, colors)):
            ax.scatter(coord[0], coord[1], coord[2], 
                      c=color, s=50, alpha=0.7)
        
        # Conexiones entre residuos consecutivos
        for i in range(len(self.current_coords) - 1):
            ax.plot([self.current_coords[i, 0], self.current_coords[i+1, 0]],
                   [self.current_coords[i, 1], self.current_coords[i+1, 1]],
                   [self.current_coords[i, 2], self.current_coords[i+1, 2]],
                   'gray', alpha=0.5, linewidth=1)
        
        # Puentes disulfuro (enlaces entre cisteínas)
        cys_positions_A = [pos for pos, aa in enumerate(self.insulin_sequence_A) if aa == 'C']
        cys_positions_B = [pos + n_chain_A for pos, aa in enumerate(self.insulin_sequence_B) if aa == 'C']
        
        # Simular algunos puentes disulfuro
        if len(cys_positions_A) >= 2:
            for i in range(0, len(cys_positions_A)-1, 2):
                pos1, pos2 = cys_positions_A[i], cys_positions_A[i+1]
                ax.plot([self.current_coords[pos1, 0], self.current_coords[pos2, 0]],
                       [self.current_coords[pos1, 1], self.current_coords[pos2, 1]],
                       [self.current_coords[pos1, 2], self.current_coords[pos2, 2]],
                       'yellow', linewidth=3, alpha=0.8)
        
        ax.set_xlabel('X (Å)')
        ax.set_ylabel('Y (Å)')
        ax.set_zlabel('Z (Å)')
        ax.set_title(f'Estructura de Insulina - Temperatura: {self.temperature:.1f}°C')
        
        # Leyenda
        ax.scatter([], [], [], c='blue', s=50, label='Cadena A')
        ax.scatter([], [], [], c='red', s=50, label='Cadena B')
        ax.plot([], [], [], 'yellow', linewidth=3, label='Puentes disulfuro')
        ax.legend()
        
        plt.tight_layout()
        return fig
    
    def plot_denaturation_curve(self):
        """
        Grafica la curva de desnaturalización vs temperatura
        """
        temperatures = np.linspace(20, 100, 100)
        denat_fractions = [self.calculate_denaturation_fraction(T) for T in temperatures]
        
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(temperatures, denat_fractions, 'b-', linewidth=2, label='Fracción desnaturalizada')
        ax.axvline(x=self.temperature, color='red', linestyle='--', 
                  label=f'Temperatura actual: {self.temperature:.1f}°C')
        ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.7, label='50% desnaturalización')
        
        ax.set_xlabel('Temperatura (°C)')
        ax.set_ylabel('Fracción desnaturalizada')
        ax.set_title('Curva de Desnaturalización de Insulina')
        ax.grid(True, alpha=0.3)
        ax.legend()
        ax.set_ylim(0, 1)
        
        return fig
    
    def create_py3dmol_viewer(self):
        """
        Crea un visor 3D interactivo usando py3Dmol
        """
        # Crear archivo PDB temporal
        pdb_content = self.generate_pdb_content()
        
        # Crear visor
        viewer = py3Dmol.view(width=800, height=600)
        viewer.addModel(pdb_content, 'pdb')
        
        # Estilo de visualización
        viewer.setStyle({'cartoon': {'color': 'spectrum'}})
        viewer.addStyle({'resn': 'CYS'}, {'stick': {'color': 'yellow'}})  # Cisteínas
        
        viewer.zoomTo()
        viewer.spin(True)
        
        return viewer
    
    def generate_pdb_content(self):
        """
        Genera contenido PDB para visualización
        """
        pdb_lines = []
        pdb_lines.append("HEADER    INSULIN DENATURATION SIMULATION")
        pdb_lines.append("REMARK    GENERATED FOR EDUCATIONAL PURPOSES")
        
        atom_id = 1
        residue_id = 1
        
        # Cadena A
        for i, aa in enumerate(self.insulin_sequence_A):
            coord = self.current_coords[i]
            line = f"ATOM  {atom_id:5d}  CA  {aa:3s} A{residue_id:4d}    {coord[0]:8.3f}{coord[1]:8.3f}{coord[2]:8.3f}  1.00 20.00           C"
            pdb_lines.append(line)
            atom_id += 1
            residue_id += 1
        
        # Cadena B
        residue_id = 1
        for i, aa in enumerate(self.insulin_sequence_B):
            coord = self.current_coords[i + len(self.insulin_sequence_A)]
            line = f"ATOM  {atom_id:5d}  CA  {aa:3s} B{residue_id:4d}    {coord[0]:8.3f}{coord[1]:8.3f}{coord[2]:8.3f}  1.00 20.00           C"
            pdb_lines.append(line)
            atom_id += 1
            residue_id += 1
        
        pdb_lines.append("END")
        
        return "\n".join(pdb_lines)

# Crear instancia del simulador
simulator = InsulinDenaturationSimulator()

def interactive_simulation():
    """
    Crea una simulación interactiva usando matplotlib widgets
    """
    # Crear figura principal con subplots
    fig = plt.figure(figsize=(16, 12))
    
    # Subplot para estructura 3D
    ax_3d = fig.add_subplot(221, projection='3d')
    
    # Subplot para curva de desnaturalización
    ax_curve = fig.add_subplot(222)
    
    # Subplot para información (texto)
    ax_info = fig.add_subplot(223)
    ax_info.axis('off')
    
    # Área para el slider
    ax_slider = plt.axes([0.2, 0.02, 0.5, 0.03])
    temp_slider = Slider(ax_slider, 'Temperatura (°C)', 20.0, 100.0, valinit=37.0, valfmt='%.1f°C')
    
    def update_plot(temperature):
        """Actualiza todos los gráficos"""
        # Actualizar estructura
        denat_fraction = simulator.update_structure(temperature)
        rmsd = simulator.calculate_rmsd()
        
        # Limpiar axes
        ax_3d.clear()
        ax_curve.clear()
        ax_info.clear()
        ax_info.axis('off')
        
        # Plot estructura 3D
        plot_3d_in_axis(ax_3d, temperature)
        
        # Plot curva de desnaturalización
        plot_curve_in_axis(ax_curve, temperature)
        
        # Mostrar información
        show_info_in_axis(ax_info, temperature, denat_fraction, rmsd)
        
        # Actualizar figura
        fig.canvas.draw()
    
    def plot_3d_in_axis(ax, temperature):
        """Plot estructura 3D en el axis dado"""
        # Colores según la secuencia
        n_chain_A = len(simulator.insulin_sequence_A)
        colors = ['blue'] * n_chain_A + ['red'] * len(simulator.insulin_sequence_B)
        
        # Plot de los puntos
        for i, (coord, color) in enumerate(zip(simulator.current_coords, colors)):
            ax.scatter(coord[0], coord[1], coord[2], c=color, s=50, alpha=0.7)
        
        # Conexiones entre residuos
        for i in range(len(simulator.current_coords) - 1):
            ax.plot([simulator.current_coords[i, 0], simulator.current_coords[i+1, 0]],
                   [simulator.current_coords[i, 1], simulator.current_coords[i+1, 1]],
                   [simulator.current_coords[i, 2], simulator.current_coords[i+1, 2]],
                   'gray', alpha=0.5, linewidth=1)
        
        # Puentes disulfuro
        cys_positions_A = [pos for pos, aa in enumerate(simulator.insulin_sequence_A) if aa == 'C']
        if len(cys_positions_A) >= 2:
            for i in range(0, len(cys_positions_A)-1, 2):
                pos1, pos2 = cys_positions_A[i], cys_positions_A[i+1]
                ax.plot([simulator.current_coords[pos1, 0], simulator.current_coords[pos2, 0]],
                       [simulator.current_coords[pos1, 1], simulator.current_coords[pos2, 1]],
                       [simulator.current_coords[pos1, 2], simulator.current_coords[pos2, 2]],
                       'yellow', linewidth=3, alpha=0.8)
        
        ax.set_xlabel('X (Å)')
        ax.set_ylabel('Y (Å)')
        ax.set_zlabel('Z (Å)')
        ax.set_title(f'Estructura Insulina - {temperature:.1f}°C')
    
    def plot_curve_in_axis(ax, current_temp):
        """Plot curva de desnaturalización"""
        temperatures = np.linspace(20, 100, 100)
        denat_fractions = [simulator.calculate_denaturation_fraction(T) for T in temperatures]
        
        ax.plot(temperatures, denat_fractions, 'b-', linewidth=2, label='Fracción desnaturalizada')
        ax.axvline(x=current_temp, color='red', linestyle='--', linewidth=2,
                  label=f'T actual: {current_temp:.1f}°C')
        ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.7, label='50% desnaturalización')
        
        ax.set_xlabel('Temperatura (°C)')
        ax.set_ylabel('Fracción desnaturalizada')
        ax.set_title('Curva de Desnaturalización')
        ax.grid(True, alpha=0.3)
        ax.legend()
        ax.set_ylim(0, 1)
    
    def show_info_in_axis(ax, temperature, denat_fraction, rmsd):
        """Muestra información en el axis"""
        info_text = f"""INFORMACIÓN DE LA SIMULACIÓN

Temperatura: {temperature:.1f}°C
Fracción desnaturalizada: {denat_fraction:.3f}
RMSD desde estructura nativa: {rmsd:.2f} Å

ESTADO DE LA PROTEÍNA:"""
        
        if temperature < 50:
            state_text = "🟢 NATIVO\nEstructura bien plegada\nActividad biológica normal"
            color = 'green'
        elif temperature < 70:
            state_text = "🟡 TRANSICIÓN\nDesnaturalización parcial\nPérdida gradual de actividad"
            color = 'orange'
        else:
            state_text = "🔴 DESNATURALIZADO\nEstructura desplegada\nPérdida total de actividad"
            color = 'red'
        
        ax.text(0.05, 0.9, info_text, transform=ax.transAxes, fontsize=11,
                verticalalignment='top', fontfamily='monospace')
        
        ax.text(0.05, 0.4, state_text, transform=ax.transAxes, fontsize=12,
                verticalalignment='top', fontweight='bold', color=color)
        
        # Leyenda de colores
        legend_text = """LEYENDA:
🔵 Cadena A de insulina
🔴 Cadena B de insulina  
🟡 Puentes disulfuro"""
        
        ax.text(0.05, 0.15, legend_text, transform=ax.transAxes, fontsize=10,
                verticalalignment='top')
    
    # Función callback para el slider
    def update_callback(val):
        temperature = temp_slider.val
        update_plot(temperature)
    
    # Conectar slider
    temp_slider.on_changed(update_callback)
    
    # Plot inicial
    update_plot(37.0)
    
    plt.suptitle('Simulación de Desnaturalización de Insulina', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.1, top=0.93)
    plt.show()

def simple_temperature_demo():
    """
    Demostración simple comparando diferentes temperaturas
    """
    temperatures = [25, 37, 60, 80, 95]
    
    fig, axes = plt.subplots(1, len(temperatures), figsize=(20, 4), 
                            subplot_kw={'projection': '3d'})
    
    for i, temp in enumerate(temperatures):
        denat_fraction = simulator.update_structure(temp)
        ax = axes[i]
        
        # Colores
        n_chain_A = len(simulator.insulin_sequence_A)
        colors = ['blue'] * n_chain_A + ['red'] * len(simulator.insulin_sequence_B)
        
        # Plot estructura
        for j, (coord, color) in enumerate(zip(simulator.current_coords, colors)):
            ax.scatter(coord[0], coord[1], coord[2], c=color, s=30, alpha=0.7)
        
        # Conexiones
        for j in range(len(simulator.current_coords) - 1):
            ax.plot([simulator.current_coords[j, 0], simulator.current_coords[j+1, 0]],
                   [simulator.current_coords[j, 1], simulator.current_coords[j+1, 1]],
                   [simulator.current_coords[j, 2], simulator.current_coords[j+1, 2]],
                   'gray', alpha=0.3, linewidth=0.5)
        
        ax.set_title(f'{temp}°C\nDesnaturalización: {denat_fraction:.2f}')
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.set_zlabel('Z')
    
    plt.suptitle('Desnaturalización de Insulina a Diferentes Temperaturas')
    plt.tight_layout()
    plt.show()

def show_py3dmol_viewer():
    """
    Muestra el visor 3D interactivo
    """
    viewer = simulator.create_py3dmol_viewer()
    return viewer.show()

# Ejecutar simulación interactiva
print("Iniciando simulación de desnaturalización de insulina...")
print("="*60)
print("INSTRUCCIONES:")
print("1. Usa el slider en la parte inferior para cambiar la temperatura")
print("2. Observa los cambios en tiempo real en la estructura 3D")
print("3. Consulta la curva de desnaturalización y la información")
print("="*60)
print("\nCaracterísticas de la simulación:")
print("- Azul: Cadena A de insulina")
print("- Rojo: Cadena B de insulina") 
print("- Amarillo: Puentes disulfuro")
print("- Temperatura de desnaturalización: ~60°C")
print("\n")

# Ejecutar simulación interactiva principal
interactive_simulation()

print("\n" + "="*60)
print("DEMOSTRACIÓN ADICIONAL:")
print("Comparación de estructuras a diferentes temperaturas")
print("="*60)

# Mostrar demostración simple
simple_temperature_demo()

# Para mostrar el visor 3D interactivo con py3Dmol, ejecuta:
print("\n" + "="*60)
print("VISOR 3D INTERACTIVO:")
print("Para ver la estructura con py3Dmol, ejecuta:")
print("viewer = simulator.create_py3dmol_viewer()")
print("viewer.show()")
print("="*60)

ModuleNotFoundError: No module named 'ipywidgets'